# AI Programming — Lecture 7
## Loss Functions

이 노트북은 Lecture 7의 핵심 흐름인

**Maximum Likelihood Estimation (MLE) → Log-Likelihood → Negative Log-Likelihood (NLL) → Loss Functions**

를 작은 숫자와 그래프로 직접 확인하기 위한 실습입니다.

이번 실습에서는 복잡한 신경망을 여러 번 학습하기보다,
**왜 특정 task에서 특정 loss function을 사용하는가**를 이해하는 데 집중합니다.

### 학습 목표

실습을 마치면 다음 내용을 설명할 수 있어야 합니다.

- likelihood, log-likelihood, NLL의 관계를 설명할 수 있습니다.
- MLE가 likelihood를 최대화하고 NLL을 최소화하는 것과 같은 문제임을 확인할 수 있습니다.
- Bernoulli distribution에서 MLE를 직접 계산할 수 있습니다.
- regression에서 MSE/MAE가 어떤 방식으로 오차를 penalize하는지 비교할 수 있습니다.
- binary classification에서 BCE를 계산할 수 있습니다.
- multiclass classification에서 CCE를 계산할 수 있습니다.
- cosine similarity를 loss 형태로 바꾸는 방법을 이해합니다.
- KL divergence가 두 probability distribution의 차이를 측정한다는 점을 확인할 수 있습니다.
- Keras의 대표적인 loss function을 직접 호출할 수 있습니다.
- 여러 loss를 결합한 composite loss를 만들 수 있습니다.

### 실습 방법

1. 셀을 위에서부터 순서대로 실행하세요.
2. 수식과 코드의 대응 관계를 확인하세요.
3. `직접 해보기`에서는 값을 수정하고 결과 변화를 관찰하세요.
4. 이번 강의에서는 **loss 값 자체보다 loss가 어떤 상황에서 작아지고 커지는지**를 이해하는 것이 중요합니다.

## 0. 라이브러리 불러오기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=5, suppress=True)

# Part I. Maximum Likelihood Estimation

## 1. Bernoulli Distribution으로 MLE 이해하기

동전의 앞면을 `1`, 뒷면을 `0`이라고 하겠습니다.

앞면이 나올 확률을 $\theta$라고 하면 Bernoulli distribution은

$$
p(x \mid \theta)
=
\theta^x(1-\theta)^{1-x}
$$

로 표현할 수 있습니다.

여러 번의 시행 결과가 i.i.d.라고 가정하면 전체 likelihood는 각 확률의 곱입니다.

$$
L(\theta)
=
\prod_{i=1}^{N}
p(x_i \mid \theta)
$$

In [ ]:
# 1: head, 0: tail
x = np.array([1, 1, 0, 1, 0, 1, 1, 0])

print("observations:", x)
print("number of trials:", len(x))
print("number of heads:", x.sum())
print("observed head ratio:", x.mean())

### 확인할 내용

이 데이터에서 앞면의 관측 비율은 `x.mean()`입니다.

MLE가 찾는 $	heta$가 이 값과 어떤 관계를 가지는지 아래에서 확인합니다.

## 2. Likelihood, Log-Likelihood, NLL

Likelihood:

$$
L(\theta)
=
\prod_i
\theta^{x_i}
(1-\theta)^{1-x_i}
$$

Log-likelihood:

$$
\ell(\theta)
=
\log L(\theta)
$$

Negative log-likelihood:

$$
\mathcal{L}(\theta)
=
-\ell(\theta)
$$

따라서

```text
Likelihood        → maximize
Log-Likelihood    → maximize
NLL               → minimize
```

는 같은 최적점을 가집니다.

In [ ]:
def bernoulli_likelihood(theta, x):
    return np.prod(
        theta ** x
        * (1 - theta) ** (1 - x)
    )

def bernoulli_log_likelihood(theta, x):
    return np.sum(
        x * np.log(theta)
        + (1 - x) * np.log(1 - theta)
    )

def bernoulli_nll(theta, x):
    return -bernoulli_log_likelihood(theta, x)


thetas = np.linspace(0.01, 0.99, 300)

likelihoods = np.array([
    bernoulli_likelihood(t, x)
    for t in thetas
])

log_likelihoods = np.array([
    bernoulli_log_likelihood(t, x)
    for t in thetas
])

nlls = np.array([
    bernoulli_nll(t, x)
    for t in thetas
])

theta_mle = thetas[np.argmax(likelihoods)]
theta_nll = thetas[np.argmin(nlls)]

print("MLE from likelihood:", theta_mle)
print("Minimum from NLL    :", theta_nll)
print("Observed head ratio :", x.mean())

In [ ]:
plt.plot(thetas, likelihoods)
plt.axvline(x.mean(), linestyle="--")
plt.xlabel("theta")
plt.ylabel("Likelihood")
plt.title("Bernoulli Likelihood")
plt.grid(alpha=0.3)
plt.show()

plt.plot(thetas, log_likelihoods)
plt.axvline(x.mean(), linestyle="--")
plt.xlabel("theta")
plt.ylabel("Log-Likelihood")
plt.title("Bernoulli Log-Likelihood")
plt.grid(alpha=0.3)
plt.show()

plt.plot(thetas, nlls)
plt.axvline(x.mean(), linestyle="--")
plt.xlabel("theta")
plt.ylabel("NLL")
plt.title("Negative Log-Likelihood")
plt.grid(alpha=0.3)
plt.show()

> ### ✅ 체크포인트
>
> - Likelihood와 log-likelihood의 최대점이 같은지 확인하세요.
> - NLL의 최소점도 같은 위치인지 확인하세요.
> - 그 위치가 관측된 앞면 비율과 거의 같은지 확인하세요.

### 직접 해보기 1 — 관측 데이터 바꾸기

아래처럼 시행 결과를 바꾸어 보세요.

```python
x = np.array([1, 1, 1, 1, 0])
```

또는

```python
x = np.array([1, 0, 0, 0, 0])
```

MLE가 관측된 앞면 비율을 따라 움직이는지 확인하세요.

## 3. 왜 Log-Likelihood를 사용하는가?

Lecture 7에서는 log를 사용하는 이유로 다음을 다룹니다.

- log는 monotonic function이므로 최대점이 바뀌지 않습니다.
- 작은 확률을 계속 곱할 때 발생하는 underflow를 줄일 수 있습니다.
- 곱셈을 덧셈으로 바꾸어 계산과 미분이 쉬워집니다.

작은 확률의 곱과 log-sum을 비교해 보겠습니다.

In [ ]:
small_probs = np.array([0.01] * 200)

product_value = np.prod(small_probs)
log_sum_value = np.sum(np.log(small_probs))

print("product of probabilities:", product_value)
print("sum of log-probabilities:", log_sum_value)

### 확인할 내용

확률을 직접 계속 곱하면 컴퓨터에서 `0`에 가까워지거나 `0`으로 표현될 수 있습니다.

반면 log를 취하면 매우 작은 확률도 안정적으로 다룰 수 있습니다.

# Part II. MLE와 Neural Network Loss의 연결

## 4. Task에 따라 Likelihood가 달라진다

Supervised learning에서 neural network는 입력 $\mathbf{x}$로부터
prediction을 만듭니다.

```text
x → neural network → z → output mapping → y_hat
```

Task에 따라 output mapping과 probability model이 달라집니다.

| Task | Output Mapping | Typical Likelihood | Common Loss |
|---|---|---|---|
| Regression | Linear | Gaussian | MSE |
| Binary Classification | Sigmoid | Bernoulli | BCE |
| Multiclass Classification | Softmax | Categorical | CCE |

즉, loss function은 단순한 임의의 공식이 아니라
**task에 맞는 likelihood를 선택한 결과로 해석할 수 있습니다.**

# Part III. Distance-Based Loss Functions

## 5. MAE와 MSE 비교

Regression에서는 prediction $\hat{y}$와 target $y$ 사이의 거리를 loss로 사용할 수 있습니다.

### MAE

$$
\mathrm{MAE}
=
\frac{1}{N}
\sum_i |y_i-\hat{y}_i|
$$

### MSE

$$
\mathrm{MSE}
=
\frac{1}{N}
\sum_i (y_i-\hat{y}_i)^2
$$

MSE는 큰 오차를 제곱하기 때문에 큰 error에 더 민감합니다.

In [ ]:
y_true = np.array([1, 2, 3, 4, 5], dtype=float)

y_pred_good = np.array([1, 2, 3, 4, 5.5], dtype=float)
y_pred_outlier = np.array([1, 2, 3, 4, 10], dtype=float)

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def mse(y_true, y_pred):
    return np.mean((y_true - y_pred) ** 2)

print("Good prediction")
print("MAE:", mae(y_true, y_pred_good))
print("MSE:", mse(y_true, y_pred_good))

print("\nPrediction with a large error")
print("MAE:", mae(y_true, y_pred_outlier))
print("MSE:", mse(y_true, y_pred_outlier))

In [ ]:
errors = np.linspace(-5, 5, 300)

mae_curve = np.abs(errors)
mse_curve = errors ** 2

plt.plot(errors, mae_curve, label="Absolute Error")
plt.plot(errors, mse_curve, label="Squared Error")
plt.xlabel("Prediction Error")
plt.ylabel("Penalty")
plt.title("MAE vs. MSE")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- 작은 error에서는 두 loss가 모두 작습니다.
- error가 커질수록 MSE가 더 빠르게 증가합니다.
- 따라서 MSE는 large error에 더 큰 penalty를 줍니다.

Lecture 7의 관점에서는 MSE를 Gaussian error assumption과 연결해서 해석할 수 있습니다.

## 6. Cosine Similarity를 Loss로 사용하기

두 vector의 cosine similarity는 방향의 유사성을 측정합니다.

$$
\cos(\theta)
=
\frac{\mathbf{y}^{\top}\hat{\mathbf{y}}}
{\|\mathbf{y}\|_2\|\hat{\mathbf{y}}\|_2}
$$

두 vector가 같은 방향이면 1, 직교하면 0, 반대 방향이면 -1입니다.

Loss로 사용하기 위해 간단히

$$
1-\cos(\theta)
$$

처럼 바꿀 수 있습니다.

In [ ]:
def cosine_similarity(a, b):
    return (a @ b) / (
        np.linalg.norm(a)
        * np.linalg.norm(b)
    )

def cosine_loss(a, b):
    return 1.0 - cosine_similarity(a, b)

a = np.array([1.0, 1.0])

b_same = np.array([2.0, 2.0])
b_orthogonal = np.array([1.0, -1.0])
b_opposite = np.array([-1.0, -1.0])

for name, b in [
    ("same", b_same),
    ("orthogonal", b_orthogonal),
    ("opposite", b_opposite),
]:
    print(
        f"{name:>10} | "
        f"cosine={cosine_similarity(a, b):.3f} | "
        f"1-cosine={cosine_loss(a, b):.3f}"
    )

# Part IV. Distribution-Based Loss Functions

## 7. Binary Cross Entropy (BCE)

Binary classification에서는 target $y \in \{0,1\}$이고,
model은 sigmoid를 통해 probability $\hat{y}$를 출력합니다.

BCE는

$$
\mathcal{L}
=
-
\left[
y\log\hat{y}
+
(1-y)\log(1-\hat{y})
\right]
$$

입니다.

In [ ]:
def binary_cross_entropy(y_true, y_prob, eps=1e-7):
    y_prob = np.clip(y_prob, eps, 1 - eps)

    return -(
        y_true * np.log(y_prob)
        + (1 - y_true) * np.log(1 - y_prob)
    )


pred_probs = np.array([
    0.99, 0.90, 0.70, 0.50, 0.10, 0.01
])

losses_when_true_1 = np.array([
    binary_cross_entropy(1, p)
    for p in pred_probs
])

for p, loss in zip(pred_probs, losses_when_true_1):
    print(
        f"y=1, predicted probability={p:.2f} "
        f"→ BCE={loss:.4f}"
    )

In [ ]:
p = np.linspace(0.01, 0.99, 300)

loss_y1 = np.array([
    binary_cross_entropy(1, value)
    for value in p
])

loss_y0 = np.array([
    binary_cross_entropy(0, value)
    for value in p
])

plt.plot(p, loss_y1, label="y = 1")
plt.plot(p, loss_y0, label="y = 0")
plt.xlabel("Predicted Probability")
plt.ylabel("BCE")
plt.title("Binary Cross Entropy")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

### 확인할 내용

- 정답이 `1`일 때 $\hat{y}$가 1에 가까울수록 loss가 작습니다.
- 정답이 `1`인데 $\hat{y}$가 0에 가까우면 loss가 매우 커집니다.
- 즉, **틀린 class에 높은 confidence를 줄수록 큰 penalty**를 받습니다.

## 8. Categorical Cross Entropy (CCE)

Multiclass classification에서 target이 one-hot vector라면

$$
\mathcal{L}
=
-
\sum_{c=1}^{C}
y_c\log\hat{y}_c
$$

를 사용합니다.

Lecture 7의 예제를 그대로 사용해 보겠습니다.

In [ ]:
y_true = np.array([0, 1, 0, 0], dtype=float)
y_pred = np.array([0.02, 0.85, 0.10, 0.03], dtype=float)

cce = -np.sum(
    y_true * np.log(y_pred)
)

print("target:", y_true)
print("prediction:", y_pred)
print("CCE:", cce)

print(
    "Equivalent to -log(correct class probability):",
    -np.log(y_pred[1])
)

### 확인할 내용

One-hot target에서는 정답 class 이외의 $y_c$가 모두 0이므로,
결국 loss에는 **정답 class의 predicted probability**가 핵심적으로 남습니다.

### 직접 해보기 2 — 정답 class probability 바꾸기

다음 예측을 비교해 보세요.

```python
[0.02, 0.95, 0.02, 0.01]
[0.10, 0.60, 0.20, 0.10]
[0.30, 0.10, 0.30, 0.30]
```

정답 class의 probability가 낮아질수록 CCE가 어떻게 변하는지 확인하세요.

## 9. Kullback-Leibler (KL) Divergence

KL divergence는 두 probability distribution $p$와 $q$의 차이를 측정합니다.

Discrete distribution에서는

$$
D_{KL}(p\|q)
=
\sum_i
p_i
\log
\frac{p_i}{q_i}
$$

입니다.

두 distribution이 같으면 0이고,
차이가 커질수록 값이 커집니다.

In [ ]:
def kl_divergence(p, q, eps=1e-12):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)

    p = np.clip(p, eps, 1.0)
    q = np.clip(q, eps, 1.0)

    return np.sum(
        p * np.log(p / q)
    )


p = np.array([0.7, 0.2, 0.1])

q_same = np.array([0.7, 0.2, 0.1])
q_close = np.array([0.5, 0.3, 0.2])
q_far = np.array([0.1, 0.2, 0.7])

print("KL(p || q_same) =", kl_divergence(p, q_same))
print("KL(p || q_close) =", kl_divergence(p, q_close))
print("KL(p || q_far)  =", kl_divergence(p, q_far))

### 확인할 내용

- `p == q`이면 KL divergence는 0입니다.
- distribution의 차이가 커질수록 KL divergence가 커집니다.
- KL divergence는 일반적인 거리(distance)처럼 symmetric하지 않습니다.

# Part V. Loss Functions in Keras

## 10. Keras Loss Object 직접 사용하기

Keras에서는 `model.compile(loss='mse')`처럼 문자열로 지정할 수도 있지만,
loss function 자체를 object로 만들어 직접 호출할 수도 있습니다.

Lecture 7에서 소개한 대표적인 loss를 일부 확인해 봅니다.

In [ ]:
import tensorflow as tf
from tensorflow.keras import losses

mse_loss = losses.MeanSquaredError()
mae_loss = losses.MeanAbsoluteError()
bce_loss = losses.BinaryCrossentropy()
cce_loss = losses.CategoricalCrossentropy()
kl_loss = losses.KLDivergence()
cos_loss = losses.CosineSimilarity()

# Regression
reg_true = tf.constant([[1.0, 2.0, 3.0]])
reg_pred = tf.constant([[1.0, 2.5, 2.0]])

print("MSE:", float(mse_loss(reg_true, reg_pred)))
print("MAE:", float(mae_loss(reg_true, reg_pred)))

# Binary classification
bin_true = tf.constant([[1.0]])
bin_pred = tf.constant([[0.8]])

print("BCE:", float(bce_loss(bin_true, bin_pred)))

# Multiclass classification
cat_true = tf.constant([[0.0, 1.0, 0.0]])
cat_pred = tf.constant([[0.1, 0.8, 0.1]])

print("CCE:", float(cce_loss(cat_true, cat_pred)))

# KL divergence
p_tf = tf.constant([[0.7, 0.2, 0.1]])
q_tf = tf.constant([[0.5, 0.3, 0.2]])

print("KL:", float(kl_loss(p_tf, q_tf)))

# Cosine similarity
a_tf = tf.constant([[1.0, 1.0]])
b_tf = tf.constant([[2.0, 2.0]])

print("Keras CosineSimilarity:", float(cos_loss(a_tf, b_tf)))

### 주의

Keras의 `CosineSimilarity` loss는 일반적인 `1 - cosine similarity`와 같은 형태가 아니라
**negative cosine similarity**를 반환합니다.

즉, Keras에서는 cosine similarity가 높을수록 loss가 더 작아지도록 부호를 반대로 둡니다.

이번 실습에서는 "같은 유사도 개념을 loss로 바꾸어 사용할 수 있다"는 점에 집중합니다.

# Part VI. Composite Loss

## 11. 여러 Loss를 결합하기

Lecture 7에서는 여러 loss를 가중합으로 결합하는 composite loss를 소개합니다.

예를 들어

$$
\mathcal{L}_{total}
=
\lambda_1\mathcal{L}_1
+
\lambda_2\mathcal{L}_2
$$

처럼 사용할 수 있습니다.

간단한 예제로 MSE와 BCE를 결합해 보겠습니다.

In [ ]:
from tensorflow.keras.losses import (
    MeanSquaredError,
    BinaryCrossentropy
)

mse = MeanSquaredError()
bce = BinaryCrossentropy()

def combined_loss(y_true, y_pred):
    return (
        0.5 * mse(y_true, y_pred)
        + 0.5 * bce(y_true, y_pred)
    )

y_true = tf.constant([[1.0], [0.0]])
y_pred = tf.constant([[0.8], [0.2]])

print(
    "combined loss:",
    float(combined_loss(y_true, y_pred))
)

## 12. Loss Weight의 영향

같은 두 loss를 사용하더라도 $\lambda$에 따라 전체 loss에서 각 항의 영향이 달라집니다.

In [ ]:
loss1 = 0.2
loss2 = 1.0

for lambda1 in [0.2, 0.5, 0.8]:
    lambda2 = 1.0 - lambda1

    total_loss = (
        lambda1 * loss1
        + lambda2 * loss2
    )

    print(
        f"lambda1={lambda1:.1f}, "
        f"lambda2={lambda2:.1f} "
        f"→ total loss={total_loss:.3f}"
    )

### 확인할 내용

Composite loss에서는 단순히 여러 loss를 더하는 것이 아니라,
각 loss가 학습에 어느 정도 영향을 주도록 할지 weight를 정해야 합니다.

# 13. 최종 실습

### 기본

1. Bernoulli observation을 바꾸고 MLE가 어떻게 변하는지 확인하세요.
2. likelihood, log-likelihood, NLL의 최적점이 같은지 확인하세요.
3. MAE와 MSE에서 large error가 있을 때 penalty 차이를 비교하세요.
4. 같은 방향, 직교, 반대 방향 vector의 cosine-based loss를 비교하세요.

### Classification

5. BCE에서 틀린 class에 높은 confidence를 줄 때 loss가 얼마나 커지는지 확인하세요.
6. CCE에서 정답 class probability를 `0.9 → 0.5 → 0.1`로 바꾸어 loss를 비교하세요.
7. one-hot label에서 CCE가 `-log(correct class probability)`와 같음을 확인하세요.

### Distribution

8. 세 개의 prediction distribution을 만들어 KL divergence를 비교하세요.

### Keras

9. `MeanSquaredError`, `BinaryCrossentropy`, `CategoricalCrossentropy`를 직접 호출하세요.
10. `model.compile(loss='mse')`에서 `'mse'`가 실제로 어떤 계산을 의미하는지 설명해 보세요.

### 도전

11. 다음 형태의 composite loss를 직접 만들어 보세요.

$$
\mathcal{L}_{total}
=
0.7\mathcal{L}_{MSE}
+
0.3\mathcal{L}_{BCE}
$$

12. weight를 `0.7/0.3 → 0.3/0.7`로 바꾸었을 때 total loss가 어떻게 달라지는지 확인하세요.

# 14. 정리

이번 실습의 핵심 흐름은 다음과 같습니다.

```text
Observed Data
    ↓
Likelihood
    ↓
Log-Likelihood
    ↓
Negative Log-Likelihood
    ↓
Loss Function
```

### MLE 관점

```text
Likelihood 최대화
= Log-Likelihood 최대화
= NLL 최소화
```

### Task별 대표적인 Loss

| Task | 대표 Loss |
|---|---|
| Regression | MSE, MAE |
| Binary Classification | BCE |
| Multiclass Classification | CCE |
| Distribution Matching | KL Divergence |
| Representation Similarity | Cosine-based Loss |

### 꼭 기억할 것

1. **Loss는 model prediction과 target 사이의 mismatch를 하나의 scalar로 요약합니다.**
2. **어떤 loss를 사용할지는 task와 output의 의미에 따라 달라집니다.**
3. **많은 supervised learning loss는 likelihood와 NLL의 관점에서 해석할 수 있습니다.**
4. **여러 목적을 동시에 학습해야 할 때는 composite loss를 사용할 수 있습니다.**